In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import os
from natsort import natsorted

import scanpy as sc
import seaborn as sns

from scroutines import basicu

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tools.sm_exceptions import ValueWarning
from tqdm import tqdm


import sys
sys.path.insert(0, '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/myvisctx/analysis_multiome/')
import lmm

In [2]:
%%time
outfigdir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/'
f = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/superdupermegaRNA_hasraw_multiome_IT.h5ad'
adata_raw = sc.read(f)
adata_raw

CPU times: user 2.23 s, sys: 16.2 s, total: 18.5 s
Wall time: 1min 31s


AnnData object with n_obs × n_vars = 89287 × 16567
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden', 'time'
    var: 'feature_types'
    layers: 'norm'

In [3]:
f = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/L23_labels_gao25_to_yoo25_knn.csv' 
df_lbl = pd.read_csv(f)
df_lbl

,label,conf
AAACCGAAGTTCCTGC-1-P21a-2023 Multiome-9-0,118_L2/3 IT CTX Glut_4,0.474420
AAACGCGCAATCATGT-1-P21a-2023 Multiome-9-0,109_L2/3 IT CTX Glut_2,1.000000
AAACGCGCAGTTGCGT-1-P21a-2023 Multiome-9-0,109_L2/3 IT CTX Glut_2,0.808598
AAACGCGCATTGTGTG-1-P21a-2023 Multiome-9-0,109_L2/3 IT CTX Glut_2,0.807582
AAAGCAAGTTGACTTC-1-P21a-2023 Multiome-9-0,118_L2/3 IT CTX Glut_4,0.603338
...,...,...
CTCACACTCCCGTTTA-1-P21DRa-2023 Multiome-10-0,110_L2/3 IT CTX Glut_2,0.801094
TAATGGTGTGGAAGGC-1-P21DRb-2023 Multiome-10-0,110_L2/3 IT CTX Glut_2,0.934366
AGCATCCCATGCATAT-1-P21DRa-2023 Multiome-10-0,110_L2/3 IT CTX Glut_2,0.746450
GGTCTTTGTTTACTTG-1-P21DRb-2023 Multiome-10-0,116_L2/3 IT CTX Glut_3,0.659059


In [4]:
adata = adata_raw[df_lbl.index].copy()
adata.obs = adata.obs.join(df_lbl)
adata.obs
adata

AnnData object with n_obs × n_vars = 6469 × 16567
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden', 'time', 'label', 'conf'
    var: 'feature_types'
    layers: 'norm'

In [5]:
adata.X.data

array([35.,  2.,  1., ...,  2.,  1.,  2.], dtype=float32)

In [6]:
adata.obs['Age'].unique()

['P21', 'P21DR']
Categories (2, object): ['P21', 'P21DR']

In [7]:
adata.obs['Sample'].unique()

['P21a', 'P21b', 'P21DRa', 'P21DRb']
Categories (4, object): ['P21DRa', 'P21DRb', 'P21a', 'P21b']

In [8]:
adata.obs['total_counts'].unique()

array([21809., 26366., 14682., ...,  7392.,  4866.,  5871.], dtype=float32)

In [9]:
clusters = np.sort(adata.obs['label'].unique())
clusters

array(['109_L2/3 IT CTX Glut_2', '110_L2/3 IT CTX Glut_2',
       '111_L2/3 IT CTX Glut_2', '116_L2/3 IT CTX Glut_3',
       '118_L2/3 IT CTX Glut_4'], dtype=object)

In [10]:
import time

In [11]:
%%time

obs_fixed1 = 'Age'
obs_fixed2 = None # 'Light'
obs_random = 'Sample'

cluster_col = 'label'

offset = 1e-2
scale = 1e4

for cluster in clusters:
    tag = f"d260303_{cluster.replace('/', '').replace(' ', '_')}"
    output = os.path.join(outfigdir, f'NRDR_DEGs_LMM_yoo25_P21_{tag}.csv')

    adatasub = adata[adata.obs[cluster_col]==cluster]
    genes = adatasub.var.index.values 

    if obs_fixed2 is None:
        obs = adatasub.obs[[obs_fixed1, obs_random]].copy()
    else:
        obs = adatasub.obs[[obs_fixed1, obs_fixed2, obs_random]].copy()
    obs = obs.dropna()
    adatasub = adatasub[obs.index]

    # mat_raw = np.array(adatasub.X.todense())
    # mat_raw = np.array(adatasub.raw.X.todense())
    mat_raw = np.array(adatasub.X.todense())
    
    # ### test
    # adatasub = adatasub[:,:20]
    # genes = genes[:20]
    # mat_raw = mat_raw[:,:20]
    # ### test

    # mat (CP10k norm)
    # mat = mat_raw/adatasub.obs['n_counts'].values.reshape(-1,1)*scale
    mat = mat_raw/adatasub.obs['total_counts'].values.reshape(-1,1)*scale

    res = lmm.run_lmm(mat, genes, obs, obs_fixed1, obs_random, output_csv=output, offset=offset)
    print(output)

(1907, 16567) (1907, 2)
(1907, 16344) (1907, 2)
(1907, 9016) (1907, 2)
156 ['Zdbf2' 'Rpl37a' 'Gm28294' 'Dbi' 'Btg2' 'Glul' 'Cnih3' 'Kcnk2' 'Pfkfb3'
 'Arl5b' 'Ptgds' 'Rnd3' 'Tnfaip6' 'Nr4a2' '2600014E21Rik' 'Pamr1' 'Cst3'
 'Cbln4' 'Atp5e' 'Rps21' 'Rpl39' 'Xist' 'Plp1' 'Il1rapl2' 'Shroom2'
 'Fabp5' 'Car2' 'Skil' 'Tiparp' '4921511C10Rik' 'S100a1' 'Gstm5' 'Gstm1'
 'Rpl34' 'Adh5' 'Gng5' '1110017D15Rik' 'Nr4a3' 'Lpar1' 'Plpp3' 'Rps8'
 'Stk40' 'Serinc2' 'Tnfrsf25' 'Sema3e' 'Rheb' 'Ost4' 'Fosl2' 'Sgsm1'
 'Tsc22d4' 'Thsd7a' 'Gm15581' 'Ptprz1' 'Pde1c' 'Suclg2' 'Slc6a1' 'Rpl32'
 'Il17ra' 'P3h3' 'Rps5' 'Fosb' 'Apoe' 'Rps19' 'Mag' 'Luzp2' 'Hs3st4'
 '4930598N05Rik' 'Samd3' 'Gja1' 'Egr2' 'S100b' 'Midn' 'Rps15' 'Phlda1'
 'Nab2' 'Cd63' 'Irs2' 'Prag1' 'Tll1' 'Gadd45gip1' 'Mt3' 'Mt2' 'Mt1'
 'Cbfa2t3' 'Fam107a' 'Rps24' 'Ndrg2' 'Adam2' 'Egr3' 'Barx2' 'Ubash3b'
 'Pts' 'Cryab' 'Sik2' 'Gm17231' 'Arid3b' 'Rec114' 'Rplp1' 'Megf11'
 'Trim71' 'Vrk2' 'Sparc' 'Rpl26' 'Per1' 'Kdm6b' '1700016P03Rik' 'Aldoc'
 'Taco1' 

100% 9016/9016 [21:06<00:00,  7.12it/s]


43 ['Gm28294' 'Btg2' 'Rnd3' 'Xist' '4921511C10Rik' 'Gstm1' '1110017D15Rik'
 'Sema3e' 'Sgsm1' 'Thsd7a' 'Il17ra' 'P3h3' 'Fosb' 'Egr2' 'Midn' 'Nab2'
 'Irs2' 'Prag1' 'Cbfa2t3' 'Egr3' 'Barx2' 'Gm17231' 'Rec114' 'Megf11'
 'Sparc' 'Per1' 'Kdm6b' '1700016P03Rik' 'Ckmt2' 'Homer1' 'Klf10' 'Arc'
 'Grasp' 'Impg2' 'Sox8' 'Dusp1' 'Cdkn1a' 'Pim1' 'Kdm5d' 'Eif2s3y' 'Uty'
 'Ddx3y' 'Dusp5']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_109_L23_IT_CTX_Glut_2.csv
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_109_L23_IT_CTX_Glut_2.csv
(2510, 16567) (2510, 2)
(2510, 16374) (2510, 2)
(2510, 9045) (2510, 2)
95 ['Eya1' 'Coq10b' 'Pth2r' 'Gm28294' 'Cdh20' 'Epb41l5' 'Btg2' 'Arl5b'
 'Fibcd1' 'Nr4a2' 'Trp53i11' 'Fbn1' 'Xist' 'Plp1' 'Il1rapl2' 'Skil'
 'Tiparp' '4921511C10Rik' 'Efna3' 'Ciart' 'Ddit4l' 'Tox' 'Gm11867'
 'Dnajb5' 'Nr4a3' 'Jun' 'Plpp3' 'Lrp8os2' 'Stk40' 'Hgf' 'Rheb' 'Fosl2'
 'Parm1' 'Pon2' 'Egr4' 'P

100% 9045/9045 [13:30<00:00, 11.17it/s]


58 ['Eya1' 'Coq10b' 'Gm28294' 'Arl5b' 'Fibcd1' 'Nr4a2' 'Trp53i11' 'Xist'
 'Il1rapl2' 'Skil' 'Tiparp' '4921511C10Rik' 'Efna3' 'Ciart' 'Tox'
 'Gm11867' 'Dnajb5' 'Nr4a3' 'Jun' 'Lrp8os2' 'Stk40' 'Rheb' 'Fosl2' 'Pon2'
 'Prkd2' 'Fosb' 'Ntn4' 'Nab2' 'Irs2' 'Cntnap4' 'Cbfa2t3' 'Sh2d4b' 'Spry2'
 'Kdm4d' 'Pde4a' 'Sik2' 'Layn' 'Jade2' 'Kdm6b' '1700016P03Rik' 'Doc2b'
 'Nxn' 'Tbc1d16' 'Dok3' 'Fst' 'Klf10' 'Angpt1' 'Trib1' 'Myh9' 'Grasp'
 'Sox8' 'Sik1' 'Tmem232' 'Eif2s3y' 'Uty' 'Ddx3y' 'Chst9' 'Rorb']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_110_L23_IT_CTX_Glut_2.csv
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_110_L23_IT_CTX_Glut_2.csv
(1550, 16567) (1550, 2)
(1550, 16229) (1550, 2)
(1550, 9005) (1550, 2)
227 ['Crispld1' 'Zdbf2' 'Pth2r' 'Vwc2l' 'Rpl37a' 'Igfbp5' 'Ptprn' 'Gm28294'
 'Bcl2' 'Dbi' 'Tmem163' 'Lemd1' 'Gpr37l1' 'Glul' 'Ier5' 'Atp1a2' 'Kcnk2'
 'Pfkfb3' 'Ptgds' '0610009E02Rik' 'R

...

58 ['Gm28294' 'Bcl2' 'Pfkfb3' 'Nr4a2' 'Bdnf' 'Xist' 'Skil' 'Tiparp' 'Sema4a'
 'Nr4a3' 'Ece1' 'Tnfrsf25' 'Rheb' 'Fosl2' 'Kctd8' 'Mn1' 'Sgsm1' 'Gm15083'
 'Fosb' 'Apoe' 'Ptpre' 'Midn' 'Gadd45b' 'Irs2' 'Prag1' 'Mast3' 'Cdyl2'
 'Cbfa2t3' 'Anxa11' 'Gm48006' 'Egr3' 'Pde4a' 'Sik2' 'Gm17231' 'Arid3b'
 'Mei4' 'Jade2' 'Kdm6b' '1700016P03Rik' 'Dusp14' 'Fmnl1' 'Pcsk1' 'Homer1'
 'Hmgcr' 'Baz1a' 'Frmd6' 'Klf10' 'A730060N03Rik' 'Scube1' 'Phf21b' 'Nr4a1'
 '2510009E07Rik' 'Etv5' 'Plcxd2' 'Sox8' 'Eif2s3y' 'Uty' 'Ddx3y']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_111_L23_IT_CTX_Glut_2.csv
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_111_L23_IT_CTX_Glut_2.csv
(172, 16567) (172, 2)
(172, 14455) (172, 2)
(172, 10120) (172, 2)
1682 ['Gm1992' '4732440D04Rik' 'St18' ... 'Csf2ra' 'Vamp7' 'CAAA01118383.1']


100% 10120/10120 [09:23<00:00, 17.96it/s]


15 ['2610203C22Rik' 'Ccdc115' 'Xist' 'Acsl1' 'Nrp1' 'Cep57' 'Pgm3'
 '1700102P08Rik' 'Slc25a38' 'Rara' 'Aanat' 'Trib2' 'Mei1' 'Grm4' 'Atl2']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_116_L23_IT_CTX_Glut_3.csv
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_116_L23_IT_CTX_Glut_3.csv
(330, 16567) (330, 2)
(330, 15248) (330, 2)
(330, 9702) (330, 2)
1577 ['Gm1992' 'Mrpl15' 'Lactb2' ... 'Eif3a' 'mt-Atp8' 'mt-Nd3']


100% 9702/9702 [09:17<00:00, 17.41it/s]


42 ['Gm28836' 'Vwc2l' 'Ralb' 'Ralgps2' 'Enkur' 'BB218582' 'Zfp661' 'Xist'
 'Phex' 'Lhfp' 'Smg5' 'Astn2' 'Cdk5rap2' 'Zfp46' 'Clnk' 'Cpeb2' 'Rnf10'
 'Ceacam2' 'E230029C05Rik' 'Lhpp' 'Akap7' 'Agpat3' 'Lrp1' 'Fam149a'
 'Gm10649' 'Csnk2a2' 'Slc9a5' 'Acin1' 'Mthfs' 'Wsb1' 'Fn3k' 'Smad5'
 'Prox2' 'Robo1' 'Thoc6' 'Metrn' 'A930015D03Rik' 'Cntnap5c' 'Tmem178'
 'Eif2s3y' 'Uty' '2310026I22Rik']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_118_L23_IT_CTX_Glut_4.csv
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_118_L23_IT_CTX_Glut_4.csv
CPU times: user 1h 6min 10s, sys: 54.6 s, total: 1h 7min 5s
Wall time: 1h 6min 54s


In [12]:
adata

AnnData object with n_obs × n_vars = 6469 × 16567
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden', 'time', 'label', 'conf'
    var: 'feature_types'
    layers: 'norm'